#### Environment Check

In [1]:
import sys
print(sys.executable)

/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/.venv/bin/python


#### Setup

In [3]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

#### Project Paths

In [4]:
LAB_DIR = Path("..").resolve()
CODE_DIR = LAB_DIR / "code"
DATA_DIR = LAB_DIR / "data"
REPORTS_DIR = LAB_DIR / "reports"

str(LAB_DIR), str(DATA_DIR)

('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab',
 '/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/data')

#### Import Course Helpers

In [5]:
import sys

sys.path.append(str(CODE_DIR))

from ingest import load_faq_data, build_index
from evaluation_utils import evaluate

#### Load Ground Truth

In [6]:
ground_truth_path = DATA_DIR / "ground_truth-new.csv"

df_ground_truth = pd.read_csv(ground_truth_path)
ground_truth = df_ground_truth.to_dict(orient="records")

len(ground_truth)

565

#### Load FAQ Documents

In [7]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm

len(documents)

113

#### Build Search Index

In [8]:
index = build_index(documents)

#### Baseline Text Search

In [9]:
def text_search(query):
    boost_dict = {
        "question": 3.0,
        "section": 0.5,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

#### Baseline Metrics

In [10]:
baseline_metrics = evaluate(
    ground_truth,
    text_search,
)

baseline_metrics

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.7734513274336283, 'mrr': 0.6204424778761058}

#### Search With Question Boost

In [11]:
def search_boost_question(query, question_boost):
    boost_dict = {
        "question": question_boost,
        "section": 0.5,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

#### Tune Question Boost

In [12]:
question_boost_results = []

for question_boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    metrics = evaluate(
        ground_truth,
        lambda query, question_boost=question_boost: search_boost_question(
            query,
            question_boost,
        ),
    )

    result = {
        "question_boost": question_boost,
        "hit_rate": metrics["hit_rate"],
        "mrr": metrics["mrr"],
    }

    question_boost_results.append(result)

df_question_boost = pd.DataFrame(question_boost_results)

df_question_boost.sort_values("mrr", ascending=False)

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

,question_boost,hit_rate,mrr
0,0.5,0.823009,0.706608
1,1.0,0.812389,0.683215
2,3.0,0.773451,0.620442
3,5.0,0.732743,0.582743
4,10.0,0.699115,0.547404


#### Search With All Boosts

In [13]:
def search_boost_all(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "answer": answer_boost,
        "section": section_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

#### Grid Search Boosts

In [14]:
grid_results = []

for question_boost in [0.5, 1.0, 2.0, 3.0]:
    for answer_boost in [0.5, 1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5, 1.0]:
            metrics = evaluate(
                ground_truth,
                lambda query,
                question_boost=question_boost,
                answer_boost=answer_boost,
                section_boost=section_boost: search_boost_all(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost,
                ),
            )

            result = {
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": metrics["hit_rate"],
                "mrr": metrics["mrr"],
            }

            grid_results.append(result)

df_grid = pd.DataFrame(grid_results)

df_grid.sort_values("mrr", ascending=False).head(10)

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

...

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

  0%|          | 0/565 [00:00<?, ?it/s]

,question,answer,section,hit_rate,mrr
78,3.0,10.0,0.5,0.920354,0.789764
52,2.0,4.0,0.1,0.923894,0.787670
76,3.0,10.0,0.1,0.916814,0.787198
28,1.0,2.0,0.1,0.922124,0.786608
29,1.0,2.0,0.2,0.913274,0.786608
4,0.5,1.0,0.1,0.913274,0.786608
53,2.0,4.0,0.2,0.922124,0.786608
54,2.0,4.0,0.5,0.911504,0.786460
8,0.5,2.0,0.1,0.918584,0.785516
33,1.0,4.0,0.2,0.918584,0.785516


#### Best Boosts

In [15]:
best_result = df_grid.sort_values("mrr", ascending=False).iloc[0]

best_result

question     3.000000
answer      10.000000
section      0.500000
hit_rate     0.920354
mrr          0.789764
Name: 78, dtype: float64

#### Tuned Text Search

In [16]:
def tuned_text_search(query):
    boost_dict = {
        "question": float(best_result["question"]),
        "answer": float(best_result["answer"]),
        "section": float(best_result["section"]),
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

#### Tuned Metrics

In [17]:
tuned_metrics = evaluate(
    ground_truth,
    tuned_text_search,
)

tuned_metrics

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.9203539823008849, 'mrr': 0.7897640117994096}

#### Save Tuning Results

In [21]:
#### Save Tuning Results

DATA_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

df_question_boost.to_csv(DATA_DIR / "question-boost-results.csv", index=False)
df_grid.to_csv(DATA_DIR / "search-tuning-results.csv", index=False)

report_path = REPORTS_DIR / "search_metrics.md"

report_lines = [
    "",
    "## Search Tuning",
    "",
    "### Baseline",
    "",
    f"- Hit Rate: {baseline_metrics['hit_rate']}",
    f"- MRR: {baseline_metrics['mrr']}",
    "",
    "### Tuned",
    "",
    f"- Question boost: {best_result['question']}",
    f"- Answer boost: {best_result['answer']}",
    f"- Section boost: {best_result['section']}",
    f"- Hit Rate: {tuned_metrics['hit_rate']}",
    f"- MRR: {tuned_metrics['mrr']}",
]

with open(report_path, "a") as f:
    f.write("\n".join(report_lines))
    f.write("\n")

report_path

PosixPath('/Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/zoomcamp-llm-2026/04-evaluation/evaluation-lab/reports/search_metrics.md')